# 14 · Bounded Worker reading in live search

This notebook adds resumable multi-move group reading to AHR v2 search. It keeps all legal moves available, preserves UNKNOWN when computation runs out, and separates forced verification visits from policy teaching counts. It never turns a local capture proof into a whole-game win.

**Inputs:** tagged definitions from notebooks 00–03 and 08; a Scout/Worker model with the existing `go2_eval_batch` contract. **Outputs:** drop-in search APIs, `reports/ahr_v2/worker_reading_validation.json`, executed fixture results and a readable search-count plot. **Expected runtime:** under five minutes, with no training. **Checkpoints:** `checkpoints/ahr-v2-reader-validation-<id>/`; production runs store the returned tree in their own full checkpoint.

**Exact resume:** import the same tagged definitions, reconstruct the unchanged model/config/root state, call `go2_search_from_state_dict(saved['active_search'], model, root, config, project=PROJECT)`, then `go2_search(..., tree=tree)`. Reader DFS stacks, per-task and total node budgets, pending inference reservations, proof facts, forced paths, and verification counters all reside in the tree. Unlimited-clock interrupted runs resume exactly; wall-clock cutoffs retain the complete state but are inherently sensitive to machine speed.

A REFUTED_WITHIN_BOUND result means capture was not forced within the specified horizon. It does not certify life, a whole-game result, or unrestricted optimality.

In [ ]:
from pathlib import Path
import nbformat, json, torch, copy, dataclasses, time, numpy as np
PROJECT=Path('/home/benjamin/workspace/ahr_go')
for name in ['00_contracts_and_recovery.ipynb','01_rules_and_replay.ipynb','02_scout_worker_network.ipynb','03_batched_puct_search.ipynb','08_tactical_constraints_and_search.ipynb']:
 for cell in nbformat.read(PROJECT/'experiments/ahr_v2'/name,4).cells:
  if cell.cell_type=='code' and 'ahr-export' in cell.metadata.get('tags',[]):exec(compile(cell.source,name,'exec'),globals())
torch.set_num_threads(2)


## A bounded question with complete legal replies

The attacker seeks to remove the specified original stones within a finite number of plies. At defender turns every legal reply, including pass and distant captures, is considered. Move ordering changes efficiency, never the set of replies. Original stones are tracked through the continuation so later reoccupation does not erase an already verified capture.

In [ ]:
GO2_READING_SEARCH_NOTEBOOK='14_worker_reading_and_search.ipynb'
GO2_READING_PROVED='PROVED_CAPTURE_WITHIN_BOUND'
GO2_READING_REFUTED='REFUTED_WITHIN_BOUND'


def go2_reading_job(state,target_points,attacker,depth=6,node_budget=64):
    points=sorted(set(int(p) for p in target_points))
    if attacker not in (-1,1) or depth<0 or node_budget<1 or not points:
        raise ValueError('Invalid bounded capture question.')
    if any(p<0 or p>=len(state.board) or state.board[p]!=-attacker for p in points):
        raise ValueError('Each original target must be a defender stone.')
    frame={'state':go2_state_to_dict(state),'remaining':int(depth),'surviving':points,
           'entered':False,'actions':[],'cursor':0,'unknown':False,'principal':[], 'incoming':None}
    return {'schema':'ahr-v2-bounded-reader-v1','attacker':int(attacker),'depth':int(depth),
            'node_budget':int(node_budget),'nodes':0,'transitions':0,'target_points':points,
            'stack':[frame],'status':'PENDING','principal_variation':[],
            'all_legal_replies':True,'full_game_optimality':False,
            'goal':'capture specified original stones within the stated number of plies'}


def _go2_reading_finish_frame(job,status,principal=()):
    completed=job['stack'].pop();line=list(principal)
    if not job['stack']:
        job['status']=status;job['principal_variation']=line;return
    parent=job['stack'][-1];action=completed['incoming']
    parent_state=go2_state_from_dict(parent['state'])
    decisive=(parent_state.to_play==job['attacker'] and status==GO2_READING_PROVED) or (parent_state.to_play!=job['attacker'] and status==GO2_READING_REFUTED)
    if decisive:
        _go2_reading_finish_frame(job,status,[action,*line]);return
    parent['unknown'] |= status=='UNKNOWN'
    if len(line)+1>len(parent['principal']):parent['principal']=[action,*line]


def _go2_reading_order(state,targets):
    # A complete legal set, ordered to inspect liberty plays and captures first.
    liberties=set()
    for p in targets:
        if state.board[p]:liberties.update(go2_group(state.board,state.size,p)[1])
    legal=go2_legal_actions(state)
    return sorted((int(a) for a in legal),key=lambda a:(a not in liberties,a==state.pass_action,a))


def go2_reading_advance(job,quantum=32,deadline=None):
    """Advance a serialized all-legal minimax stack by a bounded node quantum.

    Node/deadline exhaustion yields UNKNOWN, never a proved unrestricted loss.
    Quantum exhaustion leaves PENDING and is resumable without changing evidence.
    """
    if quantum<1:raise ValueError('A reader quantum must be positive.')
    started_nodes=job['nodes']
    while job['status']=='PENDING':
        if deadline is not None and time.monotonic()>=deadline:
            job['status']='UNKNOWN';job['cutoff']='wall_clock';break
        if job['nodes']-started_nodes>=quantum:break
        frame=job['stack'][-1];state=go2_state_from_dict(frame['state'])
        if not frame['surviving']:
            _go2_reading_finish_frame(job,GO2_READING_PROVED);continue
        if frame['remaining']==0 or state.terminal:
            _go2_reading_finish_frame(job,GO2_READING_REFUTED);continue
        if not frame['entered']:
            if job['nodes']>=job['node_budget']:
                job['status']='UNKNOWN';job['cutoff']='node_budget';break
            job['nodes']+=1;frame['entered']=True
            frame['actions']=_go2_reading_order(state,frame['surviving'])
        if frame['cursor']>=len(frame['actions']):
            status='UNKNOWN' if frame['unknown'] else (GO2_READING_REFUTED if state.to_play==job['attacker'] else GO2_READING_PROVED)
            _go2_reading_finish_frame(job,status,frame['principal']);continue
        action=frame['actions'][frame['cursor']];frame['cursor']+=1
        child=go2_step(state,action);job['transitions']+=1
        surviving=[p for p in frame['surviving'] if child.board[p]==-job['attacker']]
        job['stack'].append({'state':go2_state_to_dict(child),'remaining':frame['remaining']-1,
            'surviving':surviving,'entered':False,'actions':[],'cursor':0,'unknown':False,
            'principal':[],'incoming':action})
    return job


def go2_read_capture(state,target_points,attacker,depth=6,node_budget=64,deadline=None):
    job=go2_reading_job(state,target_points,attacker,depth,node_budget)
    while job['status']=='PENDING':go2_reading_advance(job,quantum=32,deadline=deadline)
    return {k:copy.deepcopy(v) for k,v in job.items() if k!='stack'}


## Search budgets and safe evidence

Reading shares a fixed total budget across at most four selected tasks, at the root and shallow descendants. It investigates strong candidate moves that leave a substantial group with one or two liberties, and attackable enemy groups. A proved bounded capture adds a local edge hint and a forcing variation; it never replaces the network's whole-game value. The queue stores the actual DFS cursor between search batches.

Forced visits remain real search visits, but are removed from the Scout policy target and action-support counts. This prevents a deliberately checked bad move from becoming a desirable training example.

In [ ]:
if '_GO2_READER_BASE' not in globals():
    _GO2_READER_BASE={name:globals()[name] for name in ['Go2SearchConfig','_go2_new_tree','_go2_expand','_go2_select_path','_go2_resolve_batch','go2_validate_search_tree','go2_search']}

@dataclasses.dataclass(frozen=True)
class Go2SearchConfig(_GO2_READER_BASE['Go2SearchConfig']):
    tactical_prior_weight: float = 0.15
    tactical_edge_weight: float = 0.35
    tactical_reply_reservations: int = 8
    reading_depth: int = 6
    reading_total_nodes: int = 256
    reading_task_nodes: int = 64
    reading_quantum: int = 16
    reading_max_tasks: int = 4
    reading_max_search_depth: int = 2
    reading_edge_weight: float = 0.45
    reading_wall_time_seconds: float | None = 0.05

    def __post_init__(self):
        super().__post_init__()
        if self.reading_depth<1 or min(self.reading_total_nodes,self.reading_task_nodes,self.reading_quantum,self.reading_max_tasks,self.reading_max_search_depth)<0 or self.reading_quantum<1:
            raise ValueError('Invalid Worker reading budget.')
        if self.reading_total_nodes and self.reading_max_tasks and self.reading_task_nodes<1:
            raise ValueError('An enabled reader requires a positive per-task node budget.')
        if not 0<=self.reading_edge_weight<=1 or (self.reading_wall_time_seconds is not None and self.reading_wall_time_seconds<0):
            raise ValueError('Invalid bounded reading weight/deadline.')


def _go2_search_identity(model,state,config,model_identity,context,project,game):
    if model_identity is None:
        if model is None:raise ValueError('Custom evaluator needs an immutable model identity.')
        model_identity=go2_model_identity(model)
    project=go2_project(project)
    sources={**go2_source_identity(project),**{GO2_ROOT+'/'+name:go2_code_sha(project/GO2_ROOT/name) for name in [GO2_REPAIR_SEARCH_NOTEBOOK,GO2_READING_SEARCH_NOTEBOOK]}}
    return {'schema':'ahr-go-v2-search-tree-v3-worker-reading','model':str(model_identity),'sources':sources,
            'config':dataclasses.asdict(config),'root':go2_digest(game['key'](state)),
            'context':go2_digest({} if context is None else context)}


def _go2_new_tree(identity,state,config,game):
    tree=_GO2_READER_BASE['_go2_new_tree'](identity,state,config,game)
    tree.update(reading_jobs=[],reading_facts=[],reading_nodes=0,reading_cursor=0,
                reading_line_queue=[],last_selected_verification=False)
    tree['nodes'][tree['root']]['search_depth']=0
    tree['counters'].update(verification_simulations=0,reading_tasks=0,reading_proofs=0,
                            reading_refutations=0,reading_unknown=0,reading_atomic_quanta=0)
    tree['queen']['kind']='batched_puct_with_bounded_worker_reading'
    return tree


def _go2_reading_add_job(tree,node_id,state,targets,attacker,scope,action,config):
    if len(tree['reading_jobs'])>=config.reading_max_tasks or not config.reading_total_nodes:return
    key=go2_digest([go2_state_digest(state),sorted(targets),attacker,scope,action])
    if any(j['key']==key for j in tree['reading_jobs']):return
    job=go2_reading_job(state,targets,attacker,config.reading_depth,config.reading_task_nodes)
    job.update(key=key,node_id=int(node_id),scope=scope,action=action,applied=False)
    tree['reading_jobs'].append(job);tree['counters']['reading_tasks']+=1


def _go2_expand(tree,node_id,output,config,game):
    _GO2_READER_BASE['_go2_expand'](tree,node_id,output,config,game)
    if 'identity' not in tree:return  # disposable output-validation probe
    node=tree['nodes'][node_id]
    for edge in node['edges'].values():edge.setdefault('verification_n',0)
    if not isinstance(node['state'],Go2State):return
    state=node['state'];depth=node.get('search_depth',99)
    if node_id==tree['root']:
        # Deduplicate a threatened original group and reply across root moves.
        best={}
        for a,v in node.get('tactical',{}).items():
            for reply,risk in v['reply_losses'].items():
                child=go2_step(state,a);next_state=go2_step(child,int(reply))
                removed=tuple(p for p,v0 in enumerate(state.board) if v0==state.to_play and next_state.board[p]!=state.to_play)
                key=(int(reply),removed);rank=(int(risk),node['edges'][a]['prior'],-a)
                if key not in best or rank>best[key][0]:best[key]=(rank,[a,int(reply)])
        tree['tactical_queue']=[]
        tree['reading_line_queue'].extend({'actions':line,'source':'immediate_capture_witness'} for _,line in sorted(best.values(),reverse=True)[:config.tactical_reply_reservations])
    if depth>config.reading_max_search_depth or not config.reading_total_nodes:return
    # Reserve half the task slots for genuinely encountered descendants.
    limit=max(1,config.reading_max_tasks//2) if depth==0 else config.reading_max_tasks
    if len(tree['reading_jobs'])>=limit:return
    enemies=[(len(stones),sorted(stones)) for stones,libs in go2_groups(state.board,state.size)
             if state.board[next(iter(stones))]==-state.to_play and len(libs)<=2 and len(stones)>=2]
    if enemies:
        targets=max(enemies)[1]
        _go2_reading_add_job(tree,node_id,state,targets,state.to_play,'attack',None,config)
    candidates=sorted(node['edges'],key=lambda a:(node['edges'][a]['prior'],-a),reverse=True)[:3]
    for action in candidates:
        if len(tree['reading_jobs'])>=limit:break
        if action==state.pass_action:continue
        child=go2_step(state,action)
        if child.terminal:continue
        own=[(len(stones),sorted(stones)) for stones,libs in go2_groups(child.board,child.size)
             if child.board[next(iter(stones))]==state.to_play and len(libs)<=2 and len(stones)>=2]
        if own:
            _go2_reading_add_job(tree,node_id,child,max(own)[1],-state.to_play,'candidate_group_risk',int(action),config)


def _go2_reading_root_path(tree,node_id):
    if node_id==tree['root']:return []
    frontier=[(tree['root'],[])]
    while frontier:
        parent,path=frontier.pop(0)
        for action,edge in tree['nodes'][parent]['edges'].items():
            if edge['child'] is None:continue
            if edge['child']==node_id:return [*path,action]
            if len(path)<3:frontier.append((edge['child'],[*path,action]))
    return None


def _go2_reading_apply(tree,job,config):
    if job['applied'] or job['status']=='PENDING':return
    job['applied']=True
    fact={k:copy.deepcopy(v) for k,v in job.items() if k not in ('stack','applied')}
    tree['reading_facts'].append(fact)
    tree['counters'][{GO2_READING_PROVED:'reading_proofs',GO2_READING_REFUTED:'reading_refutations','UNKNOWN':'reading_unknown'}[job['status']]]+=1
    if job['status']!=GO2_READING_PROVED:return
    node=tree['nodes'][job['node_id']];line=job['principal_variation']
    if job['scope']=='attack':
        if not line:return
        action=line[0];sign=1
    else:action=job['action'];sign=-1;line=[action,*line]
    if action not in node['edges']:return
    hint=sign*config.reading_edge_weight*math.tanh(len(job['target_points'])/4)
    edge=node['edges'][action];edge['reading_bias']=hint
    edge['tactical_bias']=edge.get('tactical_bias',0)+hint
    prefix=_go2_reading_root_path(tree,job['node_id'])
    if prefix is not None and line:
        tree['reading_line_queue'].append({'actions':[*prefix,*line], 'source':'proved_bounded_capture'})


def _go2_read_pending(tree,config):
    if not tree.get('reading_jobs'):return
    remaining=config.reading_total_nodes-tree['reading_nodes']
    deadline=None if config.reading_wall_time_seconds is None else time.monotonic()+config.reading_wall_time_seconds
    quantum=min(config.reading_quantum,max(0,remaining));advanced=0
    for index,job in enumerate(tree['reading_jobs']):
        if job['status']!='PENDING':continue
        tree['reading_cursor']=index
        if quantum<=0:
            job['status']='UNKNOWN';job['cutoff']='shared_node_budget';_go2_reading_apply(tree,job,config);continue
        before=job['nodes'];go2_reading_advance(job,quantum=quantum,deadline=deadline)
        delta=job['nodes']-before;tree['reading_nodes']+=delta;quantum-=delta;advanced+=delta
        _go2_reading_apply(tree,job,config)
        if quantum<=0:break
    tree['counters']['reading_atomic_quanta']+=int(advanced>0)


def _go2_select_path(tree,config,game,reserved=()):
    tree['last_selected_verification']=False;reserved=set(reserved)
    for item in list(tree.get('reading_line_queue',[])):
        parent=tree['root'];path=[];valid=True
        for action in item['actions']:
            node=tree['nodes'][parent]
            if node['terminal'] or not node['expanded'] or action not in node['edges']:
                valid=False
                if item in tree['reading_line_queue']:tree['reading_line_queue'].remove(item)
                break
            edge=node['edges'][action]
            if edge['child'] is None:
                child_id=tree['next_node'];tree['next_node']+=1
                tree['nodes'][child_id]=_go2_node(game['step'](node['state'],action),game);edge['child']=child_id
            child_id=edge['child'];child=tree['nodes'][child_id]
            path.append((parent,action));child['search_depth']=len(path)
            if not child['expanded'] or len(path)==len(item['actions']):
                if len(path)==len(item['actions']) and edge['n']:
                    tree['reading_line_queue'].remove(item);valid=False;break
                if child_id in reserved and not child['terminal']:valid=False;break
                tree['last_selected_verification']=True
                return path,child_id,False
            parent=child_id
        if valid and item in tree['reading_line_queue']:tree['reading_line_queue'].remove(item)
    selected=_GO2_READER_BASE['_go2_select_path'](tree,config,game,reserved)
    if selected is not None:
        for depth,(parent,action) in enumerate(selected[0],1):tree['nodes'][tree['nodes'][parent]['edges'][action]['child']]['search_depth']=depth
    return selected


def _go2_collect_batch(tree,config,game,count):
    if tree['pending']:raise AssertionError('Resolve pending inference before reserving another batch.')
    _go2_read_pending(tree,config)
    reserved=set()
    while len(tree['pending'])<count:
        selected=_go2_select_path(tree,config,game,reserved)
        if selected is None:break
        path,leaf,cutoff=selected
        if not tree['nodes'][leaf]['terminal']:reserved.add(leaf)
        _go2_reserve(tree,path,leaf)
        tree['pending'].append({'path':path,'leaf':leaf,'depth_cutoff':bool(cutoff),
            'simulation_id':tree['completed_simulations']+len(tree['pending']),
            'verification':bool(tree['last_selected_verification'])})
    if not tree['pending']:raise AssertionError('Nonterminal root must have a search simulation.')


def _go2_resolve_batch(tree,evaluator,config,game):
    records=copy.deepcopy(tree['pending'])
    _GO2_READER_BASE['_go2_resolve_batch'](tree,evaluator,config,game)
    for record in records:
        if record.get('verification'):
            tree['counters']['verification_simulations']+=1
            for parent,action in record['path']:
                edge=tree['nodes'][parent]['edges'][action]
                edge['verification_n']=edge.get('verification_n',0)+1
    go2_validate_search_tree(tree)


def go2_validate_search_tree(tree):
    _GO2_READER_BASE['go2_validate_search_tree'](tree)
    if tree.get('reading_nodes',0)>tree['config'].get('reading_total_nodes',0):raise ValueError('Worker exceeded its shared node budget.')
    for node in tree['nodes'].values():
        for edge in node['edges'].values():
            if not 0<=edge.get('verification_n',0)<=edge['n']:raise ValueError('Verification visits exceed real visits.')
    if sum(e.get('verification_n',0) for e in tree['nodes'][tree['root']]['edges'].values())!=tree['counters'].get('verification_simulations',0):
        raise ValueError('Root verification visits are inconsistent.')
    return True


def go2_search(model,state,config=None,tree=None,stop_after_simulations=None,on_batch=None,evaluator=None,model_identity=None,context=None,project=None,_game=None):
    config=Go2SearchConfig() if config is None else config
    out=_GO2_READER_BASE['go2_search'](model,state,config,tree,stop_after_simulations,on_batch,evaluator,model_identity,context,project,_game)
    tree=out['tree'];root=tree['nodes'][tree['root']]
    counts=np.asarray(out['visit_counts'],dtype=np.int64)
    verification=np.zeros_like(counts)
    for action,edge in root['edges'].items():verification[action]=edge.get('verification_n',0)
    support=counts-verification;policy=support.astype(np.float64)
    if policy.sum():
        policy/=policy.sum()
        out['action']=int(max(root['edges'],key=lambda a:(support[a],root['edges'][a]['w']/root['edges'][a]['n']+root['edges'][a].get('tactical_bias',0) if root['edges'][a]['n'] else -float('inf'),root['edges'][a]['prior'],-a)))
    out.update(policy_target=policy.tolist(),policy_target_valid=bool(support.sum()),
        policy_visit_counts=support.tolist(),verification_visit_counts=verification.tolist(),
        reading_facts=copy.deepcopy(tree['reading_facts']),reading_nodes=tree['reading_nodes'],
        reading_pending=sum(j['status']=='PENDING' for j in tree['reading_jobs']),
        policy_provenance={'kind':'search_policy','counts':'real PUCT visits excluding forced verification',
                           'local_proofs_are_not_game_outcomes':True})
    return out

Go2WorkerReadingSearchConfig=Go2SearchConfig
go2_worker_reading_search=go2_search

go2_worker_reading_search_state_dict=go2_search_state_dict
go2_worker_reading_search_from_state_dict=go2_search_from_state_dict


In [ ]:
def _go2_evaluate_nodes(tree, node_ids, evaluator, config, game):
    missing = {}
    for node_id in node_ids:
        node = tree['nodes'][node_id]
        if node['terminal'] or node['expanded']:
            continue
        key = _go2_inference_key(tree, node['state'], game)
        if key in tree['cache']:
            tree['counters']['cache_hits'] += 1
            _go2_expand(tree, node_id, tree['cache'][key], config, game)
        else:
            missing.setdefault(key, []).append(node_id)
    if not missing:
        return
    keys = list(missing)
    states = [tree['nodes'][missing[key][0]]['state'] for key in keys]
    outputs = evaluator(states)
    if len(outputs) != len(states):
        raise ValueError('Evaluator result count differs from the inference batch.')
    validated = []
    for key, output in zip(keys, outputs):
        if 'network_version' in output and str(output['network_version']) != tree['identity']['model']:
            raise ValueError('Evaluator model identity changed during an active search.')
        minimal = {'prior': np.asarray(output['prior'], dtype=np.float64).tolist(),
                   'value': float((1-config.score_value_weight)*float(output['value'])+config.score_value_weight*math.tanh(float(output.get('score',0.0))*3.0))}
        # Validate every output on a disposable tree before committing any expansion.
        probe = copy.copy(tree['nodes'][missing[key][0]])
        check_tree = {'nodes': {1: probe}, 'root': 0, 'rng': tree['rng']}
        _go2_expand(check_tree, 1, minimal, config, game)
        validated.append((key, minimal))
    tree['counters']['neural_batches'] += 1
    tree['counters']['neural_positions'] += len(states)
    for key, minimal in validated:
        tree['cache'][key] = minimal
        for node_id in missing[key]:
            _go2_expand(tree, node_id, minimal, config, game)

## Validation from legal game histories

The closed-corner fixture needs three plies; the ladder fixture needs five. The smaller horizons must not certify capture. A separate safe-group example must remain only refuted within its bound. Full CPU and CUDA search tests compare an uninterrupted tree against a fresh-kernel restore with both unfinished reader work and pending inference reservations. The original six-stone failure is diagnostic development evidence only; it is never added to training.

In [ ]:
def go2_worker_reading_validation(project=None):
    import uuid, subprocess, sys
    project=go2_project(project);started=time.monotonic();checks={};fixtures=[]
    closed=go2_replay(go2_initial_state(size=5,komi=0),[2,0,7,1,10,25,11,25])
    ladder=go2_replay(go2_initial_state(size=5,komi=0),[5,6,1,25,10,25,7,11,16,12,13,17])
    safe=go2_replay(go2_initial_state(size=5,komi=0),[24,12])
    for name,state,targets,horizon in [('closed_corner',closed,[0,1],3),('ladder',ladder,[6,11,12,17],5),('safe_center',safe,[12],3)]:
        checks[name+'_history_replays']=go2_verify_replay(state)
        before=time.monotonic();proof=go2_read_capture(state,targets,1,depth=horizon,node_budget=2048)
        shallow=go2_read_capture(state,targets,1,depth=1 if name!='ladder' else 3,node_budget=2048)
        expected=GO2_READING_REFUTED if name=='safe_center' else GO2_READING_PROVED
        checks[name+'_bounded_result']=proof['status']==expected
        checks[name+'_shallow_no_false_capture']=shallow['status']==GO2_READING_REFUTED
        fixtures.append({'name':name,'depth':horizon,'status':proof['status'],'nodes':proof['nodes'],
                         'principal_variation':proof['principal_variation'],'elapsed_seconds':time.monotonic()-before})
    # Independently establish the closed-corner three-ply strategy by checking
    # EVERY legal defender reply after attack A2, then replaying capture B2.
    after_attack=go2_step(closed,5);all_replies=go2_legal_actions(after_attack)
    independently_forced=True
    for reply in all_replies:
        defended=go2_step(after_attack,int(reply));captured=go2_try_step(defended,6)
        independently_forced &= captured is not None and all(captured.board[p]!=-1 for p in [0,1])
    checks['independent_all_legal_three_ply_strategy']=bool(independently_forced) and len(all_replies)>10
    cutoff=go2_read_capture(ladder,[6,11,12,17],1,depth=5,node_budget=1)
    deadline=go2_read_capture(ladder,[6,11,12,17],1,depth=5,node_budget=2048,deadline=time.monotonic()-1)
    checks['node_cutoff_is_UNKNOWN']=cutoff['status']=='UNKNOWN'
    checks['clock_cutoff_is_UNKNOWN']=deadline['status']=='UNKNOWN'
    job=go2_reading_job(ladder,[6,11,12,17],1,depth=5,node_budget=64)
    go2_reading_advance(job,quantum=1);paused=copy.deepcopy(job)
    while job['status']=='PENDING':go2_reading_advance(job,quantum=3)
    while paused['status']=='PENDING':go2_reading_advance(paused,quantum=3)
    checks['reader_stack_exact_resume']=go2_nested_equal(job,paused)
    model=Go2Network(Go2NetConfig(board_size=5,width=16,blocks=1));model.eval()
    production_cfg=Go2SearchConfig(simulations=128,batch_size=8,reading_wall_time_seconds=None)
    out=go2_search(model,ladder,production_cfg,project=project)
    checks['production_contains_multimove_capture_fact']=any(f['status']==GO2_READING_PROVED and f['depth']>=5 and len(f['principal_variation'])>=5 for f in out['reading_facts'])
    checks['production_all_legal_actions_retained']=set(out['tree']['nodes'][0]['edges'])==set(go2_legal_actions(ladder))
    checks['forced_counts_are_separate']=sum(out['verification_visit_counts'])>0 and sum(out['visit_counts'])==128
    checks['policy_counts_exclude_forced_verification']=out['policy_visit_counts']==(np.asarray(out['visit_counts'])-np.asarray(out['verification_visit_counts'])).tolist()
    checks['policy_target_normalizes_support']=np.allclose(np.asarray(out['policy_target']),np.asarray(out['policy_visit_counts'])/sum(out['policy_visit_counts']))
    checks['bounded_proofs_do_not_replace_global_value']=out['search_status']=='UNKNOWN' and abs(out['value'])<=1 and all(not f['full_game_optimality'] for f in out['reading_facts'])
    checks['shared_reader_budget_retained']=out['reading_nodes']<=production_cfg.reading_total_nodes
    devices=['cpu']+(['cuda:1' if torch.cuda.device_count()>1 else 'cuda:0'] if torch.cuda.is_available() else [])
    recovery=[]
    for device in devices:
        local_model=copy.deepcopy(model).to(device);local_model.eval()
        cfg=Go2SearchConfig(simulations=32,batch_size=4,reading_quantum=1,reading_wall_time_seconds=None,seed=2026100814)
        full=go2_search(local_model,ladder,cfg,project=project)['tree']
        partial=go2_search(local_model,ladder,cfg,stop_after_simulations=4,project=project)['tree']
        _go2_collect_batch(partial,cfg,_go2_search_game(),4)
        pending=any(j['status']=='PENDING' and len(j['stack'])>1 for j in partial['reading_jobs'])
        checks[device+'_pending_reader_and_inference_saved']=pending and bool(partial['pending'])
        token=uuid.uuid4().hex[:10];run_dir=project/'checkpoints'/('ahr-v2-reader-validation-'+token)
        payload={'schema':GO2_SCHEMA,'identity':partial['identity'],'model':local_model.state_dict(),
                 'optimizer':{},'scheduler':{},'scaler':{},'rng':go2_capture_rng(cuda_devices=[int(device.split(':')[1])] if device.startswith('cuda') else []),
                 'sampler':{},'global_step':4,'config':{'network':dataclasses.asdict(local_model.config),'search':dataclasses.asdict(cfg),'device':device},
                 'metrics':{'simulations':4,'reading_nodes':partial['reading_nodes']},'root_state':go2_state_to_dict(ladder),
                 'active_search':go2_search_state_dict(partial)}
        with go2_checkpoint_lock(run_dir):entry=go2_save_checkpoint(run_dir,payload,best=True,resume_cell='14: tagged exports; exact saved model/config/root; go2_search_from_state_dict')
        child_path=project/'reports/ahr_v2/fixtures'/('worker-reading-resume-'+token+'.ipynb')
        result_path=child_path.with_suffix('.json')
        child_source="""from pathlib import Path
import nbformat, torch, json, dataclasses
project=Path(PROJECT_LITERAL)
for name in ['00_contracts_and_recovery.ipynb','01_rules_and_replay.ipynb','02_scout_worker_network.ipynb','03_batched_puct_search.ipynb','08_tactical_constraints_and_search.ipynb','14_worker_reading_and_search.ipynb']:
 for cell in nbformat.read(project/'experiments/ahr_v2'/name,4).cells:
  if cell.cell_type=='code' and 'ahr-export' in cell.metadata.get('tags',[]):exec(compile(cell.source,name,'exec'),globals())
torch.set_num_threads(2)
saved,entry=go2_load_checkpoint(project/'checkpoints'/RUN_LITERAL)
model=Go2Network(Go2NetConfig(**saved['config']['network'])).to(saved['config']['device']);model.load_state_dict(saved['model']);model.eval()
state=go2_state_from_dict(saved['root_state']);cfg=Go2SearchConfig(**saved['config']['search'])
tree=go2_search_from_state_dict(saved['active_search'],model,state,cfg,project=project)
assert tree['pending'] and any(j['status']=='PENDING' for j in tree['reading_jobs'])
out=go2_search(model,state,cfg,tree=tree,project=project)
payload=go2_search_state_dict(out['tree']);payload.pop('payload_sha256');payload.pop('elapsed_seconds')
go2_atomic_json(Path(RESULT_LITERAL),{'digest':go2_digest(payload),'completed':out['completed_simulations'],'reading_nodes':out['reading_nodes']})
print('Restored pending reader and inference on',saved['config']['device'])
""".replace('PROJECT_LITERAL',repr(str(project))).replace('RUN_LITERAL',repr(run_dir.name)).replace('RESULT_LITERAL',repr(str(result_path)))
        child_nb=nbformat.v4.new_notebook(cells=[nbformat.v4.new_code_cell(child_source)],metadata={'kernelspec':{'display_name':'AHR Go','language':'python','name':'ahr-go'}})
        nbformat.write(child_nb,child_path)
        completed=subprocess.run([sys.executable,'-m','jupyter','nbconvert','--execute','--to','notebook','--inplace','--ExecutePreprocessor.timeout=180',str(child_path)],cwd=project,capture_output=True,text=True,timeout=240)
        if completed.returncode:raise RuntimeError(completed.stdout+'\n'+completed.stderr)
        fresh=json.loads(result_path.read_text());baseline=go2_search_state_dict(full);baseline.pop('payload_sha256');baseline.pop('elapsed_seconds')
        checks[device+'_fresh_kernel_search_exact_resume']=fresh['digest']==go2_digest(baseline) and fresh['completed']==32
        broken=copy.deepcopy(payload['active_search']);broken['reading_nodes']+=1
        try:go2_search_from_state_dict(broken,local_model,ladder,cfg,project=project);rejected=False
        except ValueError:rejected=True
        checks[device+'_corrupt_search_refuses_restart']=rejected
        recovery.append({'device':device,'run_dir':str(run_dir.relative_to(project)),'checkpoint':entry['file'],'sha256':entry['sha256'],'fixture':str(child_path.relative_to(project))})
        del local_model
        if device.startswith('cuda'):torch.cuda.empty_cache()
    # The preserved development failure has a long precursor; the finite
    # reader is checked at the already unsavable stage, without training on it.
    old_path=project/'reports/ahr_v2/evaluations/ahr-v2-repair-20261007-8h-final-dev-752/report.json'
    previous=json.loads(old_path.read_text());actions=previous['games'][0]['state']['actions']
    failed=go2_replay(go2_initial_state(),actions[:20])
    targets=max((sorted(st) for st,libs in go2_groups(failed.board,failed.size) if failed.board[next(iter(st))]==failed.to_play),key=len)
    prior_failure=go2_read_capture(failed,targets,-failed.to_play,depth=6,node_budget=128)
    checks['old_six_stone_loss_verified_within_bound']=len(targets)==6 and prior_failure['status']==GO2_READING_PROVED
    report={'schema':'ahr-v2-worker-reading-validation-v1','status':'PASS' if all(checks.values()) else 'FAIL',
            'checks':checks,'fixtures':fixtures,'sources':out['tree']['identity']['sources'],
            'search':{'simulations':128,'reading_nodes':out['reading_nodes'],'verification_simulations':sum(out['verification_visit_counts']),
                      'policy_support_simulations':sum(out['policy_visit_counts']),'selected_action':out['action'],'reading_facts':out['reading_facts'],
                      'root_counts':out['visit_counts'],'verification_counts':out['verification_visit_counts'],'support_counts':out['policy_visit_counts']},
            'recovery':recovery,'old_failure':{'before_ply':21,'target_stones':len(targets),'bounded_result':prior_failure['status'],'nodes':prior_failure['nodes'],
             'limitation':'The six-ply horizon does not certify the earlier sixteen-ply trap; stronger learned priors/value and deeper selective reading still matter.'},
            'runtime_seconds':time.monotonic()-started,'claim':'Bounded capture, live integration, honest targets and exact recovery validated; no playing-strength claim.'}
    go2_atomic_json(project/'reports/ahr_v2/worker_reading_validation.json',report)
    if not all(checks.values()):raise AssertionError({k:v for k,v in checks.items() if not v})
    return report


In [ ]:
validation=go2_worker_reading_validation(PROJECT)
print(validation['status'], '—', len(validation['checks']), 'checks in', round(validation['runtime_seconds'],2), 'seconds')
for fixture in validation['fixtures']:
 print(fixture['name'],':',fixture['status'],'at',fixture['depth'],'plies;',fixture['nodes'],'reader nodes')
print('Production fixture:', validation['search']['verification_simulations'],'verification visits and', validation['search']['policy_support_simulations'],'policy-support visits')


## Real search visits and useful policy evidence

The chart separates deliberate verification from ordinary PUCT support. Both contribute to the actual search tree; only ordinary support contributes to the policy target. These are fixture results, not match wins.

In [ ]:
import matplotlib.pyplot as plt
counts=np.asarray(validation['search']['root_counts']);verification=np.asarray(validation['search']['verification_counts']);support=np.asarray(validation['search']['support_counts'])
actions=np.flatnonzero(counts)
labels=[('pass' if a==25 else f"{'ABCDE'[a%5]}{5-a//5}") for a in actions]
fig,ax=plt.subplots(figsize=(10,4))
ax.bar(labels,support[actions],label='PUCT policy support',color='#2468a2')
ax.bar(labels,verification[actions],bottom=support[actions],label='Forced verification',color='#d28b25')
ax.set(title='128-visit legal-history ladder fixture',xlabel='Root move',ylabel='Committed search visits')
ax.legend(frameon=False);ax.spines[['top','right']].set_visible(False);ax.grid(axis='y',alpha=.2);ax.set_axisbelow(True)
fig.tight_layout();fig.savefig(PROJECT/'reports/ahr_v2/worker_reading_counts.png',dpi=150)
plt.show()


## Reading limits

A proof answers a specific finite capture question under the exact legal history. A refuted bound does not mean a group is alive. UNKNOWN does not become a loss label. The default 50 ms reader-quantum deadline prevents unresolved exhaustive reading from dominating a real-time move. For exact deterministic recovery tests, deadlines are disabled.

The preserved loss is recognized when the six-stone group is already unsavable. Its earlier long trap remains beyond this small reader's guaranteed horizon. Better teacher data and calibrated Scout/Worker learning remain necessary; this notebook alone does not establish improvement against GNU Go.